# MobileBEV trên KITTI (Google Colab)

Notebook đọc ba archive KITTI trực tiếp từ
`MyDrive/KITTI_DATASET_ZIP`, giải nén vào ổ local Colab với thanh tiến trình,
prepare dữ liệu, train và đánh giá. Checkpoint/kết quả được lưu bền vững trên
Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

%cd /content
!test -d Lidar || git clone --branch mobileBEV-architecture --single-branch https://github.com/danhyoyo/Lidar.git
%cd /content/Lidar
!git branch --show-current
!git log -1 --oneline

## Cấu hình

Chỉ cần đổi các giá trị trong cell dưới. `PHYSICAL_BATCH_SIZE` là nút chỉnh
chính nếu GPU thiếu bộ nhớ.

In [ ]:
from pathlib import Path
import json
import os
import torch

REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/mobilebev_artifacts")

VARIANT = "A0"  # B0, A0, A1, A2, A3 hoặc A4
SEED = 42
PRECISION = "fp32"  # Đổi thành "bf16" nếu GPU hỗ trợ BF16
PHYSICAL_BATCH_SIZE = 16
ACCUMULATION_STEPS = 1
NUM_WORKERS = max(0, min(8, (os.cpu_count() or 1) - 1))
EPOCHS = 100

## Cài thư viện và kiểm tra GPU

In [ ]:
%cd /content/Lidar
%pip install -q shapely onnx tqdm
!command -v mbuffer >/dev/null || (apt-get -qq update && apt-get -qq install -y mbuffer)
if _exit_code:
    raise RuntimeError("Không cài được mbuffer")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Runtime > Change runtime type.")
print("GPU:", torch.cuda.get_device_name(0))

## Giải nén và prepare KITTI

Archive được stream từ Google Drive qua `mbuffer` RAM rồi vào `tar`, không tạo
bản copy local hàng chục GiB. `mbuffer` hiển thị tốc độ/buffer, còn `tqdm`
hiển thị tiến trình giải nén theo số file.

In [ ]:
archives = {
    "velodyne": ("*.bin", 7481),
    "label_2": ("*.txt", 7481),
    "calib": ("*.txt", 7481),
}
BUFFER_SIZE = "1G"  # Giảm xuống 512M nếu runtime ít RAM
BUFFER_PREFILL = 80  # Bắt đầu giải nén khi buffer đầy 80%
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

for folder, (pattern, expected_count) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    extracted_dir = RAW_KITTI_ROOT / "training" / folder
    extracted_count = sum(1 for _ in extracted_dir.glob(pattern))

    if extracted_count != expected_count:
        if not archive.is_file():
            raise FileNotFoundError(f"Không tìm thấy archive: {archive}")
        print(f"Stream {archive.name} qua buffer {BUFFER_SIZE}")
        file_suffix = pattern.removeprefix("*")
        !set -o pipefail; mbuffer -m "{BUFFER_SIZE}" -s 1M -P {BUFFER_PREFILL} -i "{archive}" | tar --no-same-owner -xvf - -C "{RAW_KITTI_ROOT}" | grep --line-buffered -F "{file_suffix}" | python3 -m tqdm --total {expected_count} --unit files --desc "Giải nén {folder}" > /dev/null
        if _exit_code:
            raise RuntimeError(f"Giải nén thất bại: {archive}")

    actual_count = sum(1 for _ in extracted_dir.glob(pattern))
    if actual_count != expected_count:
        raise RuntimeError(f"{folder}: {actual_count} file, cần {expected_count}")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (
    sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481
    and sum(1 for _ in label_dir.glob("*.txt")) == 7481
    and (PROCESSED_DATASET_DIR / "train.txt").is_file()
    and (PROCESSED_DATASET_DIR / "val.txt").is_file()
)

if not dataset_ready:
    %cd /content/Lidar
    !python3 tools/kitti_training_pipeline/prepare_kitti.py \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --output-root "{PROCESSED_DATASET_DIR}" \
      --config-output "{REPO_DIR / 'data/kitti/generated_kitti.json'}" \
      --train-ids "{REPO_DIR / 'splits/kitti/train.txt'}" \
      --val-ids "{REPO_DIR / 'splits/kitti/val.txt'}" \
      --pointcloud-mode symlink \
      --overwrite
    if _exit_code:
        raise RuntimeError("prepare_kitti.py thất bại")

pointcloud_count = sum(1 for _ in pointcloud_dir.glob("*.bin"))
label_count = sum(1 for _ in label_dir.glob("*.txt"))
assert pointcloud_count == label_count == 7481
assert (PROCESSED_DATASET_DIR / "train.txt").is_file()
assert (PROCESSED_DATASET_DIR / "val.txt").is_file()

print(f"Raw KITTI: {RAW_KITTI_ROOT}")
print(f"Processed: {PROCESSED_DATASET_DIR}")
print(f"Frames: {pointcloud_count}")

## Kiểm tra code

In [ ]:
%cd /content/Lidar
!MPLCONFIGDIR=/tmp/mobilebev-mpl python3 tests/test_mobile_bev.py
if _exit_code:
    raise RuntimeError("Test MobileBEV thất bại")

## Chọn variant và config

- `B0`: baseline loss.
- `A0`: UWAG + CoordAtt + augmentation.
- `A1`–`A4`: các biến thể MobileBEV.

In [ ]:
CONFIGS = {
    "A0": "configs/kitti/kitti_uwag_coordatt_aug.json",
    "A1": "configs/kitti/mobilebev/a1_legacy35_center3d.json",
    "A2": "configs/kitti/mobilebev/a2_rich8_center3d.json",
    "A3": "configs/kitti/mobilebev/a3_legacy35_sgfpn_center3d.json",
    "A4": "configs/kitti/mobilebev/a4_rich8_sgfpn_center3d.json",
}

if VARIANT == "B0":
    source_config = REPO_DIR / CONFIGS["A0"]
    baseline_config = REPO_DIR / "configs/kitti/baseline_original.json"
    config = json.loads(source_config.read_text())
    config["loss"]["name"] = "baseline"
    baseline_config.write_text(json.dumps(config, indent=2) + "\n")
    CONFIG = str(baseline_config)
elif VARIANT in CONFIGS:
    CONFIG = str(REPO_DIR / CONFIGS[VARIANT])
else:
    raise ValueError(f"VARIANT không hợp lệ: {VARIANT}")

RUN_NAME = f"mobilebev_{VARIANT.lower()}_seed{SEED}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("Variant:", VARIANT)
print("Config:", CONFIG)
print("Run:", RUN_NAME)

## Smoke test

Chạy ngắn trước để phát hiện lỗi pipeline hoặc thiếu bộ nhớ.

In [ ]:
%cd /content/Lidar
!python3 tools/kitti_training_pipeline/train.py \
  --config "{CONFIG}" \
  --detector-root detector \
  --output-root "{ARTIFACT_ROOT}" \
  --run-name "{RUN_NAME}_smoke" \
  --device cuda \
  --precision "{PRECISION}" \
  --seed {SEED} \
  --epochs 1 \
  --physical-batch-size {PHYSICAL_BATCH_SIZE} \
  --accumulation-steps {ACCUMULATION_STEPS} \
  --max-train-batches 8 \
  --max-val-batches 4 \
  --num-workers {NUM_WORKERS}
if _exit_code:
    raise RuntimeError("Smoke test thất bại; giảm PHYSICAL_BATCH_SIZE nếu GPU hết bộ nhớ")

## Train full

In [ ]:
%cd /content/Lidar
!python3 tools/kitti_training_pipeline/train.py \
  --config "{CONFIG}" \
  --detector-root detector \
  --output-root "{ARTIFACT_ROOT}" \
  --run-name "{RUN_NAME}" \
  --device cuda \
  --precision "{PRECISION}" \
  --seed {SEED} \
  --epochs {EPOCHS} \
  --physical-batch-size {PHYSICAL_BATCH_SIZE} \
  --accumulation-steps {ACCUMULATION_STEPS} \
  --num-workers {NUM_WORKERS}
if _exit_code:
    raise RuntimeError("Training thất bại")

## Chọn checkpoint

`B0`/`A0` dùng checkpoint tốt nhất theo validation loss. `A1`–`A4` chọn lại
theo 3D mAP trên validation set.

In [ ]:
%cd /content/Lidar
SELECT_3D_CHECKPOINT = VARIANT in {"A1", "A2", "A3", "A4"}

if SELECT_3D_CHECKPOINT:
    !python3 tools/kitti_training_pipeline/select_checkpoint.py \
      --checkpoint-dir "{ARTIFACT_ROOT / RUN_NAME / 'checkpoints'}" \
      --config "{CONFIG}" \
      --detector-root detector \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --split splits/kitti/val.txt \
      --output-dir "{ARTIFACT_ROOT / RUN_NAME / 'selected_3d'}" \
      --device cuda
    if _exit_code:
        raise RuntimeError("Chọn checkpoint theo 3D mAP thất bại")
    CHECKPOINT = ARTIFACT_ROOT / RUN_NAME / "selected_3d/best.pt"
else:
    CHECKPOINT = ARTIFACT_ROOT / RUN_NAME / "selected/best.pt"

if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
print("Checkpoint:", CHECKPOINT)

## Đánh giá

Đặt `RUN_EVALUATION = False` nếu chỉ cần train.

In [ ]:
%cd /content/Lidar
RUN_EVALUATION = True

if RUN_EVALUATION:
    !python3 tools/kitti_training_pipeline/evaluate_kitti_bev.py \
      --name "{RUN_NAME}" \
      --backend pytorch \
      --model "{CHECKPOINT}" \
      --config "{CONFIG}" \
      --detector-root detector \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --split splits/kitti/val.txt \
      --output "{ARTIFACT_ROOT / RUN_NAME / 'evaluation.json'}" \
      --device cuda \
      --warmup-frames 10
    if _exit_code:
        raise RuntimeError("Đánh giá thất bại")